In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
pip install -q tensorflow_probability

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import numpy as np
import tensorflow as tf
import tensorflow_probability as tfp
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

2026-03-02 08:05:06.808302: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772438707.200976      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772438707.368664      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772438708.284877      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772438708.284922      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772438708.284924      55 computation_placer.cc:177] computation placer alr

In [4]:
train_dir = "/kaggle/input/datasets/amimulahasanrofik/alz-b-1100/alzheimer_new_11/alzheimer_new"
external_test_dir = "/kaggle/input/datasets/amimulahasanrofik/bra-alz-8/alzheimer_new"

IMG_SIZE = (224,224)
BATCH_SIZE = 32
NUM_CLASSES = 8
AUTOTUNE = tf.data.AUTOTUNE

In [5]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("Classes:", class_names)

Found 8800 files belonging to 8 classes.
Using 7040 files for training.


I0000 00:00:1772438747.133981      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 8800 files belonging to 8 classes.
Using 1760 files for validation.
Classes: ['MildDemented', 'Moderate Dementia', 'NonDemented', 'Very mild Dementia', 'glioma', 'meningioma', 'notumor', 'pituitary']


In [6]:
def preprocess(images, labels):
    images = preprocess_input(images)  # ResNet normalization
    labels = tf.one_hot(labels, NUM_CLASSES)
    return images, labels

train_ds = train_ds.map(preprocess, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.map(preprocess, num_parallel_calls=AUTOTUNE)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [7]:
data_aug = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

In [8]:
def mixup(batch1, batch2, alpha=0.2):
    images1, labels1 = batch1
    images2, labels2 = batch2

    lam = tfp.distributions.Beta(alpha, alpha).sample([BATCH_SIZE])
    lam = tf.reshape(lam, (BATCH_SIZE,1,1,1))

    mixed_images = images1 * lam + images2 * (1 - lam)

    lam_label = tf.reshape(lam[:,0,0,0], (BATCH_SIZE,1))
    mixed_labels = labels1 * lam_label + labels2 * (1 - lam_label)

    return mixed_images, mixed_labels

train_ds_mix = tf.data.Dataset.zip((train_ds, train_ds.shuffle(1000)))
train_ds_mix = train_ds_mix.map(lambda x,y: mixup(x,y),
                                num_parallel_calls=AUTOTUNE)
train_ds_mix = train_ds_mix.prefetch(AUTOTUNE)

In [15]:
def build_model():

    inputs = layers.Input(shape=(224,224,3))
    x = data_aug(inputs)

    # ---------------- ResNet50 ----------------
    base_model = ResNet50(
        weights="imagenet",
        include_top=False,
        input_tensor=x
    )

    base_model.trainable = True
    for layer in base_model.layers[:100]:
        layer.trainable = False

    cnn_feat = layers.GlobalAveragePooling2D()(base_model.output)

    # ---------------- ViT Patch Embedding ----------------
    patch_size = 16
    projection_dim = 128
    num_patches = (224//patch_size)**2

    patches = layers.Conv2D(
        filters=projection_dim,
        kernel_size=patch_size,
        strides=patch_size
    )(x)

    patches = layers.Reshape((num_patches, projection_dim))(patches)

    # ---------------- Transformer Blocks ----------------
    for _ in range(4):

        # LayerNorm + Attention
        x1 = layers.LayerNormalization(epsilon=1e-6)(patches)
        attn = layers.MultiHeadAttention(
            num_heads=8,
            key_dim=projection_dim,
            dropout=0.1
        )(x1, x1)

        x2 = layers.Add()([patches, attn])  # residual 1

        # LayerNorm + MLP
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)

        mlp = layers.Dense(256, activation="gelu")(x3)
        mlp = layers.Dropout(0.2)(mlp)
        mlp = layers.Dense(projection_dim)(mlp)  # 🔥 back to 128

        patches = layers.Add()([x2, mlp])  # residual 2

    # ---------------- GRU over tokens ----------------
    gru_out = layers.GRU(256)(patches)

    # ---------------- Fusion ----------------
    x = layers.Concatenate()([cnn_feat, gru_out])
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    return keras.Model(inputs, outputs)

In [16]:
model = build_model()

lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-4,
    decay_steps=1000
)

model.compile(
    optimizer=keras.optimizers.Adam(lr_schedule),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 224, 224,  │          0 │ input_layer_4[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ sequential[3][0]  │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c

 Total params: 27,681,032 (105.59 MB)

 Trainable params: 23,804,808 (90.81 MB)

 Non-trainable params: 3,876,224 (14.79 MB)

In [17]:
callbacks = [
    keras.callbacks.EarlyStopping(
        patience=8,
        restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        save_best_only=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        factor=0.3,
        patience=4
    )
]

In [ ]:
history = model.fit(
    train_ds_mix,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks
)

Epoch 1/50


I0000 00:00:1772438986.305017     128 cuda_dnn.cc:529] Loaded cuDNN version 91002


220/220 ━━━━━━━━━━━━━━━━━━━━ 124s 302ms/step - accuracy: 0.4760 - loss: 1.9475 - val_accuracy: 0.2114 - val_loss: 2.3653 - learning_rate: 8.8526e-05
Epoch 2/50
220/220 ━━━━━━━━━━━━━━━━━━━━ 69s 272ms/step - accuracy: 0.7154 - loss: 1.3421 - val_accuracy: 0.2631 - val_loss: 2.5670 - learning_rate: 5.9369e-05
Epoch 3/50
220/220 ━━━━━━━━━━━━━━━━━━━━ 69s 271ms/step - accuracy: 0.7976 - loss: 1.1490 - val_accuracy: 0.2455 - val_loss: 2.7177 - learning_rate: 2.5912e-05
Epoch 4/50
220/220 ━━━━━━━━━━━━━━━━━━━━ 69s 272ms/step - accuracy: 0.8370 - loss: 1.0849 - val_accuracy: 0.2568 - val_loss: 2.9056 - learning_rate: 3.5112e-06
Epoch 5/50
220/220 ━━━━━━━━━━━━━━━━━━━━ 69s 271ms/step - accuracy: 0.8295 - loss: 1.0867 - val_accuracy: 0.2568 - val_loss: 3.0058 - learning_rate: 0.0000e+00
Epoch 6/50
220/220 ━━━━━━━━━━━━━━━━━━━━ 73s 271ms/step - accuracy: 0.8457 - loss: 1.0593 - val_accuracy: 0.2625 - val_loss: 2.9357 - learning_rate: 0.0000e+00
Epoch 7/50
 70/220 ━━━━━━━━━━━━━━━━━━━━ 36s 245ms/step -

In [ ]:
model.save("resnet50_vit_gru_final.keras")

In [ ]:
external_ds = tf.keras.preprocessing.image_dataset_from_directory(
    external_test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

external_ds = external_ds.map(preprocess)
external_ds = external_ds.prefetch(AUTOTUNE)

results = model.evaluate(external_ds)
print("Unknown Dataset Accuracy:", results[1])